# SinLlama focused Sinhala QA v3

Select **Runtime ? Change runtime type ? T4 GPU**, then select **Runtime ? Run all**.
Upload `sigiriya_sinllama_focused_runtime_v3.zip` when prompted. The ZIP is in
`results/sinllama_focused_comparison_v3/`. To rebuild it, run
`python -B -m scripts.sinllama_focused_comparison_v3 bundle --output results/v3_rebuild`
from `Sigiriya-Hybrid-GraphRAG`.

A uses the v2 decomposed legacy prompt and original cleanup. B uses the focused Sinhala prompt, and C removes chunk IDs and section numbers from the rendered evidence. All Q03 runs share Arm A?s decoding and stopping rules. The JSON records an additional cleanup-only replay of A?s completion for comparison with B and C. Q01 and Q05 rerun their verified v2 profiles. Review each topic separately and fill the human assessment columns in the downloaded CSV.

In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2

In [ ]:
from io import BytesIO
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

import torch
from google.colab import files

assert torch.cuda.is_available(), "Select a GPU runtime before running the experiment."
print(torch.cuda.get_device_name())
bundle_name = "sigiriya_sinllama_focused_runtime_v3.zip"
uploaded = files.upload()
assert bundle_name in uploaded, f"Upload {bundle_name}"
workspace = Path(mkdtemp(prefix="sigiriya_focused_v3_", dir="/content"))
with ZipFile(BytesIO(uploaded[bundle_name])) as archive:
    archive.extractall(workspace)

In [ ]:
import os
import subprocess
import sys

name = "sinllama_focused_comparison_v3"
output = workspace / "results" / name
subprocess.run(
    [sys.executable, "-B", "-m", f"scripts.{name}", "gpu", "--output", str(output)],
    cwd=workspace, env={**os.environ, "PYTHONIOENCODING": "utf-8"}, check=True,
)

In [ ]:
import json
import sys

sys.path.insert(0, str(workspace))
from scripts.sinllama_diagnostic_v1 import EVIDENCE, load_packet
from scripts.sinllama_reporting import print_report

report = json.loads((output / f"{name}_gpu.json").read_text(encoding="utf-8"))
rows, _ = load_packet(EVIDENCE)
assert len(report["results"]) == 8
print_report(report["results"], questions={qid: row["question"] for qid, row in rows.items()})

In [ ]:
from shutil import make_archive

archive = make_archive(str(workspace / f"{name}_outputs"), "zip", output)
files.download(archive)